# Whisper LoRA Fine-tuning

### 목적

* 사전 학습된 Whisper 모델에 LoRA를 적용하여, AIHUB 186 복지 분야 콜센터 상담 음성 데이터에 맞게 파인튜닝
* 전체 모델을 다시 학습하지 않고 일부 Adapter 파라미터만 학습
* GPU 메모리와 저장 공간을 줄이면서 도메인 적응 실험 가능

#### 데이터 구성
* 전처리가 완료된 Hugging Face Dataset 사용
  - `train`: 모델 학습
  - `development`: 학습 중간 성능 확인
  - `validation`: 학습 완료 후 최종 평가
* Dataset 포함 정보
  - `file_id`
  - `duration_sec`
  - `input_features`
  - `labels`

#### 수행 과정

1. 학습 설정 및 실행 폴더 생성
2. Dataset과 기반 Whisper 모델 로드
3. LoRA Adapter 적용
4. `train` 데이터로 학습
5. `development` 데이터로 중간 WER/CER 평가
6. `validation` 데이터로 최종 성능 평가
7. Adapter, Processor, 학습 이력 및 메타데이터 저장

#### 저장 결과
* 각 실행마다 고유한 `RUN_DIR` 생성
  - `lora_adapter/`: 학습된 LoRA Adapter
  - `processor/`: Processor 및 Tokenizer
  - `checkpoints/`: 학습 중간 체크포인트
  - `metadata.json`: 모델 및 실험 정보
  - `dataset_manifest.json`: 학습 데이터 정보
  - `training_args.json`: 학습 설정
  - `train_history.csv`: 학습 로그
  - `metrics.json`: 최종 평가 결과
  - `validation_detail_with_path.csv`: 파일별 인식 결과
  - `SUCCESS` / `FAILED`: 실행 상태

#### 평가 지표
- WER, CER, Macro WER / CER, 학습 및 평가 시간, Validation RTF 등

#### 활용
* 생성된 LoRA Adapter는 ASR 평가 노트북에서 `peft_lora` 모델로 불러와 기본 모델과 성능 비교 가능

### 1. 패키지 설치

In [ ]:
pip install -U transformers datasets accelerate peft jiwer pandas tqdm

### 2. 라이브러리 로드

In [ ]:
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import hashlib, inspect, json, os, platform, random, re, shutil, socket, sys, time, traceback, types, uuid

import numpy as np
import pandas as pd
import torch
import datasets, transformers, peft

from datasets import load_from_disk
from jiwer import wer, cer
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

from IPython.display import Audio, display

### 3. 파인튜닝 설정

In [ ]:
# 실행 정보
EXPERIMENTER, EXPERIMENT_TAG = "yroh", "lora-r32"
DATASET_NAME = "aihub186"

# 모델·데이터
MODEL_ID = "openai/whisper-tiny"
HF_DATASET_DIR = Path("/home/data/expr/week2/04-asr_aihub/hf_dataset_whisper-tiny")
ASR_TSV = Path("/home/data/expr/week2/03-data_prepare_aihub/asr_dataset.tsv")
RESULTS_ROOT = Path("/home/data/expr/week2/06-asr_finetune_results")

# Whisper
LANGUAGE, TASK = "ko", "transcribe"
MAX_NEW_TOKENS, NUM_BEAMS, DO_SAMPLE = 128, 1, False
USE_FP16, USE_SAFETENSORS = True, True

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# 학습
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 128, 128
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE, WEIGHT_DECAY = 2e-5, 0.01
WARMUP_STEPS, MAX_STEPS = 20, 100
EVAL_STEPS, SAVE_STEPS, LOGGING_STEPS = 25, 25, 10
LR_SCHEDULER_TYPE, MAX_GRAD_NORM = "cosine", 1.0
SAVE_TOTAL_LIMIT = 2

# 평가
RUN_BASELINE_VALIDATION = False
BASELINE_MAX_SAMPLES = None       # None이면 전체, 일부면 59 등
FINAL_MAX_SAMPLES = None        # None이면 전체 validation
VALIDATION_BATCH_SIZE = 64
SEED = 42

# 출력
OVERWRITE_CHECKPOINTS = False

### 4. 공통 함수

In [ ]:
def safe_slug(value):
    value = re.sub(r"[^0-9A-Za-z가-힣._+-]+", "_", str(value).strip())
    return value.strip("._-") or "unknown"


def write_json_file(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)


def read_json_file(path):
    path = Path(path)
    if not path.is_file():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    if not path.is_file():
        return None

    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_text(value):
    return "" if value is None else str(value).strip()


def safe_wer(reference, hypothesis):
    try:
        return float(wer([safe_text(reference)], [safe_text(hypothesis)]))
    except Exception:
        return np.nan


def safe_cer(reference, hypothesis):
    try:
        return float(cer([safe_text(reference)], [safe_text(hypothesis)]))
    except Exception:
        return np.nan


def dataset_info(ds, path):
    durations = list(map(float, ds["duration_sec"])) if "duration_sec" in ds.column_names else []
    total_seconds = float(sum(durations))

    return {
        "path": str(path),
        "num_rows": len(ds),
        "columns": ds.column_names,
        "fingerprint": getattr(ds, "_fingerprint", None),
        "total_seconds": total_seconds,
        "total_hours": total_seconds / 3600,
    }


def decode_reference(label_ids, tokenizer):
    label_ids = np.asarray(label_ids, dtype=np.int64).copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    return safe_text(tokenizer.decode(label_ids, skip_special_tokens=True))

def get_dataset_category(dataset_dir):
    parts = Path(dataset_dir).parts
    start = next((i for i, part in enumerate(parts) if part.startswith("hf_dataset_")), None)
    category_parts = parts[start + 1:] if start is not None else []
    return " > ".join(category_parts) or "전체"


### 5. 설정 및 데이터 경로 검증

In [ ]:
if not EXPERIMENTER.strip():
    raise ValueError("EXPERIMENTER가 필요합니다.")
if not EXPERIMENT_TAG.strip():
    raise ValueError("EXPERIMENT_TAG가 필요합니다.")
if not MODEL_ID.strip():
    raise ValueError("MODEL_ID가 필요합니다.")
if not HF_DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset 루트 없음: {HF_DATASET_DIR}")
if not ASR_TSV.is_file():
    raise FileNotFoundError(f"ASR TSV 없음: {ASR_TSV}")

SPLIT_DIRS = {name: HF_DATASET_DIR / name for name in ["train", "development", "validation"]}
missing_splits = [name for name, path in SPLIT_DIRS.items() if not path.is_dir()]

if missing_splits:
    raise FileNotFoundError(f"Dataset split 없음: {missing_splits}")

DATASET_CATEGORY = get_dataset_category(HF_DATASET_DIR)

print("MODEL :", MODEL_ID)
print("DATA  :", HF_DATASET_DIR)
print("DATASET_CATEGORY  :", DATASET_CATEGORY)
print("SPLITS:", SPLIT_DIRS)

### 6. 고유 학습 실행 폴더 생성

In [ ]:
started_at = datetime.now().astimezone()
MODEL_NAME = MODEL_ID.split("/")[-1]

RUN_ID = "_".join([
    started_at.strftime("%Y%m%dT%H%M%S%z"),
    safe_slug(EXPERIMENTER),
    safe_slug(EXPERIMENT_TAG),
    uuid.uuid4().hex[:8],
])

RUN_DIR = (
    RESULTS_ROOT
    / safe_slug(DATASET_NAME)
    / safe_slug(MODEL_NAME)
    / started_at.strftime("%Y-%m-%d")
    / RUN_ID
)

RUN_DIR.mkdir(parents=True, exist_ok=False)

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "lora_adapter"
PROCESSOR_DIR = RUN_DIR / "processor"

RUNNING_FILE = RUN_DIR / "RUNNING"
SUCCESS_FILE = RUN_DIR / "SUCCESS"
FAILED_FILE = RUN_DIR / "FAILED"

write_json_file(RUNNING_FILE, {
    "run_id": RUN_ID,
    "status": "running",
    "started_at": started_at.isoformat(),
})

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)

### 7. 실행 환경 저장

In [ ]:
cuda_available = torch.cuda.is_available()

environment = {
    "hostname": socket.gethostname(),
    "platform": platform.platform(),
    "python_version": sys.version,
    "libraries": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "peft": peft.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "cuda": {
        "available": cuda_available,
        "torch_cuda_version": torch.version.cuda,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES", ""),
        "device_count": torch.cuda.device_count() if cuda_available else 0,
        "device_names": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ] if cuda_available else [],
    },
}

write_json_file(RUN_DIR / "environment.json", environment)

### 8. Dataset 읽기

In [ ]:
ds_train = load_from_disk(str(SPLIT_DIRS["train"]))
ds_development = load_from_disk(str(SPLIT_DIRS["development"]))
ds_validation = load_from_disk(str(SPLIT_DIRS["validation"]))

required_columns = {"file_id", "duration_sec", "input_features", "labels"}

for name, ds in {
    "train": ds_train,
    "development": ds_development,
    "validation": ds_validation,
}.items():
    missing = required_columns - set(ds.column_names)

    if missing:
        raise ValueError(f"{name} Dataset 열 누락: {sorted(missing)}")
    if not len(ds):
        raise ValueError(f"{name} Dataset이 비어 있습니다.")

print(ds_train)
print(ds_development)
print(ds_validation)

### 9. Dataset 메타데이터 저장

In [ ]:
dataset_manifest = {
    "dataset_name": DATASET_NAME,
    "source_tsv": str(ASR_TSV),
    "source_tsv_sha256": file_sha256(ASR_TSV),
    "hf_dataset_root": str(HF_DATASET_DIR),
    "splits": {
        "train": dataset_info(ds_train, SPLIT_DIRS["train"]),
        "development": dataset_info(ds_development, SPLIT_DIRS["development"]),
        "validation": dataset_info(ds_validation, SPLIT_DIRS["validation"]),
    },
}

write_json_file(RUN_DIR / "dataset_manifest.json", dataset_manifest)

display(pd.DataFrame({
    name: {
        "rows": info["num_rows"],
        "hours": info["total_hours"],
        "fingerprint": info["fingerprint"],
    }
    for name, info in dataset_manifest["splits"].items()
}).T)

### 10. 모델·Processor 로드

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.float16 if DEVICE.type == "cuda" and USE_FP16 else torch.float32
MODEL_DTYPE = torch.float32

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK,
)

#model = AutoModelForSpeechSeq2Seq.from_pretrained(
#    MODEL_ID,
#    torch_dtype=MODEL_DTYPE,
#    low_cpu_mem_usage=True,
#    use_safetensors=USE_SAFETENSORS,
#)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    use_safetensors=USE_SAFETENSORS,
)

model.config.use_cache = False
model.config.suppress_tokens = []

model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.num_beams = NUM_BEAMS
model.generation_config.do_sample = DO_SAMPLE
model.generation_config.max_length = MAX_NEW_TOKENS

model = model.to(DEVICE)

print("DEVICE:", DEVICE)
print("DTYPE :", MODEL_DTYPE)
print("MODEL :", type(model).__name__)

### 11. 기본 모델 메타데이터 저장

In [ ]:
base_model_metadata = {
    "run_id": RUN_ID,
    "created_at": started_at.isoformat(),
    "experimenter": EXPERIMENTER,
    "experiment_tag": EXPERIMENT_TAG,
    "dataset_name": DATASET_NAME,
    "model": {
        "base_model_id": MODEL_ID,
        "base_model_name": MODEL_NAME,
        "fine_tuning_method": "LoRA",
        "task": "automatic-speech-recognition",
        "language": LANGUAGE,
    },
    "lora": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "bias": "none",
        "target_modules": LORA_TARGET_MODULES,
        "task_type": "SEQ_2_SEQ_LM",
    },
    "paths": {
        "run_dir": str(RUN_DIR),
        "checkpoint_dir": str(CHECKPOINT_DIR),
        "adapter_dir": str(ADAPTER_DIR),
        "processor_dir": str(PROCESSOR_DIR),
        "dataset_dir": str(HF_DATASET_DIR),
        "source_tsv": str(ASR_TSV),
    },
}

write_json_file(RUN_DIR / "metadata.json", base_model_metadata)

### 12. Data Collator 및 평가 함수

In [ ]:
@dataclass
class WhisperDataCollator:
    processor: object

    def __call__(self, features):
        inputs = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(inputs, return_tensors="pt")

        label_inputs = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_inputs, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        decoder_start_id = self.processor.tokenizer.bos_token_id
        if labels.shape[1] and torch.all(labels[:, 0] == decoder_start_id):
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = WhisperDataCollator(processor)


def compute_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    hypotheses = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    references = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": float(wer(references, hypotheses)),
        "cer": float(cer(references, hypotheses)),
    }

### 13. LoRA 적용

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=LORA_TARGET_MODULES,
)

lora_model = get_peft_model(model, lora_config)

lora_model.generation_config.language = LANGUAGE
lora_model.generation_config.task = TASK
lora_model.generation_config.num_beams = NUM_BEAMS
lora_model.generation_config.do_sample = DO_SAMPLE

lora_model.print_trainable_parameters()

### 14. Whisper PEFT Forward 패치

In [ ]:
def patched_forward(
    self,
    input_features=None,
    input_ids=None,
    attention_mask=None,
    inputs_embeds=None,
    decoder_input_ids=None,
    decoder_attention_mask=None,
    decoder_inputs_embeds=None,
    labels=None,
    **kwargs,
):
    kwargs.pop("input_ids", None)
    kwargs.pop("attention_mask", None)
    kwargs.pop("inputs_embeds", None)

    return self.base_model(
        input_features=input_features,
        decoder_input_ids=decoder_input_ids,
        decoder_attention_mask=decoder_attention_mask,
        decoder_inputs_embeds=decoder_inputs_embeds,
        labels=labels,
        **kwargs,
    )


lora_model.forward = types.MethodType(patched_forward, lora_model)

### 15. TrainingArguments 생성

In [ ]:
training_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "max_steps": MAX_STEPS,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "max_grad_norm": MAX_GRAD_NORM,
    "fp16": DEVICE.type == "cuda" and USE_FP16,
    "eval_steps": EVAL_STEPS,
    "save_strategy": "steps",
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "logging_steps": LOGGING_STEPS,
    "predict_with_generate": True,
    "generation_max_length": MAX_NEW_TOKENS,
    "remove_unused_columns": True,
    "label_names": ["labels"],
    "report_to": [],
    "seed": SEED,
}

strategy_key = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    else "evaluation_strategy"
)

training_kwargs[strategy_key] = "steps"
training_args = Seq2SeqTrainingArguments(**training_kwargs)

write_json_file(RUN_DIR / "training_args.json", training_args.to_dict())

### 16. Trainer 생성

In [ ]:
trainer_kwargs = {
    "model": lora_model,
    "args": training_args,
    "train_dataset": ds_train,
    "eval_dataset": ds_development,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = processor
else:
    trainer_kwargs["tokenizer"] = processor

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Trainer ready.")

### 17. 선택적 Baseline Validation 평가

In [ ]:
baseline_metrics = None

if RUN_BASELINE_VALIDATION:
    baseline_dataset = (
        ds_validation.select(range(min(BASELINE_MAX_SAMPLES, len(ds_validation))))
        if BASELINE_MAX_SAMPLES
        else ds_validation
    )

    baseline_metrics = trainer.evaluate(
        eval_dataset=baseline_dataset,
        metric_key_prefix="baseline_validation",
    )

    write_json_file(RUN_DIR / "baseline_metrics.json", baseline_metrics)
    print(baseline_metrics)

### 18. LoRA 학습 실행

In [ ]:
try:
    training_started_at = datetime.now().astimezone()
    train_result = trainer.train()
    training_finished_at = datetime.now().astimezone()

except Exception as exc:
    failed_data = {
        "run_id": RUN_ID,
        "status": "failed",
        "failed_at": datetime.now().astimezone().isoformat(),
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "traceback": traceback.format_exc(),
    }

    write_json_file(FAILED_FILE, failed_data)

    if RUNNING_FILE.exists():
        RUNNING_FILE.unlink()

    raise

print(train_result)

### 19. 학습 이력 및 Trainer 상태 저장

In [ ]:
train_history_df = pd.DataFrame(trainer.state.log_history)
train_history_df.to_csv(RUN_DIR / "train_history.csv", index=False, encoding="utf-8-sig")

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("History rows:", len(train_history_df))
display(train_history_df.tail(10))

### 20. LoRA Adapter 및 Processor 저장

In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
PROCESSOR_DIR.mkdir(parents=True, exist_ok=True)

lora_model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(PROCESSOR_DIR))

print("Adapter  :", ADAPTER_DIR)
print("Processor:", PROCESSOR_DIR)

### 21. Development 최종 평가

In [ ]:
development_metrics = trainer.evaluate(
    eval_dataset=ds_development,
    metric_key_prefix="development",
)

write_json_file(RUN_DIR / "development_metrics.json", development_metrics)
display(pd.DataFrame([development_metrics]).T)

### 22. Validation 추론 함수

In [ ]:
@torch.inference_mode()
def evaluate_dataset(model, dataset, batch_size=8, max_samples=None):
    model.eval()

    selected = (
        dataset.select(range(min(max_samples, len(dataset))))
        if max_samples
        else dataset
    )

    detail_rows, references, hypotheses = [], [], []
    started = time.perf_counter()

    for start in tqdm(range(0, len(selected), batch_size), desc="Validation"):
        examples = [selected[i] for i in range(start, min(start + batch_size, len(selected)))]

        input_features = torch.stack([
            torch.as_tensor(ex["input_features"], dtype=MODEL_DTYPE)
            for ex in examples
        ]).to(DEVICE)

        generated_ids = model.generate(
            input_features=input_features,
            language=LANGUAGE,
            task=TASK,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=DO_SAMPLE,
        )

        batch_hypotheses = processor.tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        for ex, hypothesis in zip(examples, batch_hypotheses):
            reference = decode_reference(ex["labels"], processor.tokenizer)
            hypothesis = safe_text(hypothesis)

            references.append(reference)
            hypotheses.append(hypothesis)

            detail_rows.append({
                "file_id": ex["file_id"],
                "duration_sec": float(ex["duration_sec"]),
                "reference": reference,
                "hypothesis": hypothesis,
                "wer": safe_wer(reference, hypothesis),
                "cer": safe_cer(reference, hypothesis),
            })

    elapsed = time.perf_counter() - started
    total_audio_seconds = sum(row["duration_sec"] for row in detail_rows)

    metrics = {
        "n": len(detail_rows),
        "wer": float(wer(references, hypotheses)),
        "cer": float(cer(references, hypotheses)),
        "macro_wer": float(np.nanmean([row["wer"] for row in detail_rows])),
        "macro_cer": float(np.nanmean([row["cer"] for row in detail_rows])),
        "elapsed_sec": elapsed,
        "total_audio_seconds": total_audio_seconds,
        "real_time_factor": elapsed / max(total_audio_seconds, 1e-9),
    }

    return metrics, pd.DataFrame(detail_rows)

### 23. Validation 최종 평가

In [ ]:
validation_metrics, validation_detail_df = evaluate_dataset(
    lora_model,
    ds_validation,
    batch_size=VALIDATION_BATCH_SIZE,
    max_samples=FINAL_MAX_SAMPLES,
)

validation_detail_df.to_csv(
    RUN_DIR / "validation_detail.csv",
    index=False,
    encoding="utf-8-sig",
)

write_json_file(RUN_DIR / "validation_metrics.json", validation_metrics)

display(pd.DataFrame([validation_metrics]).T)
display(validation_detail_df.head(10))

### 24. 원본 음성 경로 결합

In [ ]:
source_df = pd.read_csv(
    ASR_TSV,
    sep="\t",
    encoding="utf-8-sig",
    dtype={"file_id": str, "audio_path": str, "transcript": str, "dataset_type": str},
)

source_df.columns = source_df.columns.str.replace("\ufeff", "", regex=False).str.strip()
source_df["file_id"] = source_df["file_id"].fillna("").astype(str).str.strip()

source_map_df = source_df.drop_duplicates("file_id", keep="first")

validation_detail_df = validation_detail_df.merge(
    source_map_df[["file_id", "audio_path", "transcript", "dataset_type"]],
    on="file_id",
    how="left",
).rename(columns={
    "transcript": "source_transcript",
    "dataset_type": "source_dataset_type",
})

validation_detail_df.to_csv(
    RUN_DIR / "validation_detail_with_path.csv",
    index=False,
    encoding="utf-8-sig",
)

### 25. 전체 학습 결과 저장

In [ ]:
finished_at = datetime.now().astimezone()

metrics = {
    "run_id": RUN_ID,
    "started_at": started_at.isoformat(),
    "training_started_at": training_started_at.isoformat(),
    "training_finished_at": training_finished_at.isoformat(),
    "finished_at": finished_at.isoformat(),
    "baseline_validation": baseline_metrics,
    "train": train_result.metrics,
    "development": development_metrics,
    "validation": validation_metrics,
}

write_json_file(RUN_DIR / "metrics.json", metrics)

### 26. 실행 완료 및 실험 목록 갱신

In [ ]:
completion_data = {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "status": "success",
    "started_at": started_at.isoformat(),
    "finished_at": finished_at.isoformat(),
    "experimenter": EXPERIMENTER,
    "experiment_tag": EXPERIMENT_TAG,
    "dataset_name": DATASET_NAME,
    "dataset_category": DATASET_CATEGORY,
    "dataset_dir": str(HF_DATASET_DIR),
    "base_model_id": MODEL_ID,
    "fine_tuning_method": "LoRA",
    "adapter_dir": str(ADAPTER_DIR),
    "processor_dir": str(PROCESSOR_DIR),
    "development_wer": development_metrics.get("development_wer"),
    "development_cer": development_metrics.get("development_cer"),
    "validation_wer": validation_metrics["wer"],
    "validation_cer": validation_metrics["cer"],
    "validation_rtf": validation_metrics["real_time_factor"],
}

write_json_file(SUCCESS_FILE, completion_data)

if RUNNING_FILE.exists():
    RUNNING_FILE.unlink()

EXPERIMENTS_MD = RESULTS_ROOT / "FINETUNE_EXPERIMENTS.md"

if not EXPERIMENTS_MD.exists():
    EXPERIMENTS_MD.write_text(
        "# ASR LoRA Fine-tuning Experiments\n\n"
        "| Finished | Run ID | User | Dataset | Category | Base model | LoRA r | Steps | Dev WER | Dev CER | Valid WER | Valid CER | Result |\n"
        "|---|---|---|---|---|---:|---:|---:|---:|---:|---:|---:|---|\n",
        encoding="utf-8",
    )

result_link = (RUN_DIR / "metrics.json").relative_to(RESULTS_ROOT).as_posix()

with EXPERIMENTS_MD.open("a", encoding="utf-8") as f:
    f.write(
        f"| {finished_at.strftime('%Y-%m-%d %H:%M:%S')} "
        f"| {RUN_ID} | {EXPERIMENTER} | {DATASET_NAME} | {DATASET_CATEGORY} | {MODEL_ID} "
        f"| {LORA_R} | {MAX_STEPS} "
        f"| {development_metrics.get('development_wer', np.nan):.6f} "
        f"| {development_metrics.get('development_cer', np.nan):.6f} "
        f"| {validation_metrics['wer']:.6f} "
        f"| {validation_metrics['cer']:.6f} "
        f"| [metrics]({result_link}) |\n"
    )

print("=" * 80)
print(f"학습 완료 | RUN_ID: {RUN_ID}")
print(f"RUN_DIR : {RUN_DIR}")
print(f"ADAPTER : {ADAPTER_DIR}")
print(f"DEV     : WER={development_metrics.get('development_wer')} CER={development_metrics.get('development_cer')}")
print(f"VALID   : WER={validation_metrics['wer']:.6f} CER={validation_metrics['cer']:.6f}")
print("=" * 80)

### 27. 오류가 큰 Validation 샘플 재생

In [ ]:
audio_map = (
    pd.read_csv(ASR_TSV, sep="\t", encoding="utf-8-sig", dtype=str)
    .rename(columns=lambda c: c.replace("\ufeff", "").strip())
    .assign(file_id=lambda x: x["file_id"].fillna("").astype(str).str.strip())
    .drop_duplicates("file_id")
    .set_index("file_id")["audio_path"]
    .to_dict()
)


REVIEW_COUNT = 10
review_df = validation_detail_df.sort_values(["cer", "wer"], ascending=False).head(REVIEW_COUNT)

for i, row in enumerate(review_df.itertuples(index=False), 1):
    audio_path = Path(audio_map.get(row.file_id, ""))

    print("=" * 100)
    print(f"[{i}/{len(review_df)}] {row.file_id}")
    print(f"DURATION: {row.duration_sec:.4f} sec | CER: {row.cer:.6f} | WER: {row.wer:.6f}")
    print("REF :", row.reference)
    print("HYP :", row.hypothesis)
    print("PATH:", audio_path)

    display(Audio(filename=str(audio_path))) if audio_path.is_file() else print("[ERROR] 음성 파일을 찾을 수 없습니다.")